# BartaGuard: transformer fine-tuning on BTTC

Fine-tunes two MIT-licensed checkpoints on the leak-free BTTC splits made by `bartaguard.data`
(near-duplicate groups never cross splits): `sagorsarker/bangla-bert-base` and `FacebookAI/xlm-roberta-base`.
Model selection and the abstention threshold use the validation split only; the test split and the
hand-written challenge set are scored once at the end.

Attach the dataset with `bttc_splits.csv` and `challenge_set.csv`, turn on a GPU and internet, then Run All.


In [ ]:
!mkdir -p /kaggle/working/src/bartaguard

In [ ]:
%%writefile /kaggle/working/src/bartaguard/__init__.py
"""BartaGuard data, models and safety rules."""


In [ ]:
%%writefile /kaggle/working/src/bartaguard/text.py
"""Text normalisation and featurisation shared by training, the API and the phone apps.

Everything here is deliberately simple and explicit so the Swift (iOS) and
Kotlin (Android) ports in apps/ can reproduce it exactly; golden test vectors
in ml/artifacts/ondevice_golden.json check that they do.

Why not sklearn's default tokenizer: Python's ``\\w`` does not match Bangla
vowel signs (Unicode Mn/Mc), so the default pattern splits words like "কিনতে"
into pieces. We treat the whole Bengali block U+0980–U+09FF as word characters.
"""
from __future__ import annotations

import re
import unicodedata

BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
_WORD = re.compile(r"[ঀ-৿a-z0-9]+")
_SPACE = re.compile(r"\s+")
_NUM = re.compile(r"[0-9]+")


def normalize(text: str) -> str:
    """NFC, lowercase, Bangla digits -> ASCII, digit runs -> '0', collapse spaces."""
    t = unicodedata.normalize("NFC", str(text)).lower().translate(BN_DIGITS)
    t = _NUM.sub("0", t)
    return _SPACE.sub(" ", t).strip()


def word_tokens(text: str) -> list:
    """Unigrams + bigrams of Bangla/Latin/digit words (on normalised text)."""
    w = _WORD.findall(normalize(text))
    return w + [f"{a} {b}" for a, b in zip(w, w[1:])]


def char_ngrams(text: str, lo: int = 2, hi: int = 4) -> list:
    """sklearn 'char_wb'-style n-grams: per word, padded with one space each side."""
    out = []
    for word in normalize(text).split(" "):
        if not word:
            continue
        p = f" {word} "
        for n in range(lo, hi + 1):
            out.extend(p[i:i + n] for i in range(len(p) - n + 1))
    return out


def group_key(text: str) -> str:
    """Key for exact/template duplicates: normalised text with URLs/masks/punctuation removed."""
    t = normalize(text)
    t = re.sub(r"https?://\S+|xxxxx", " ", t)
    return " ".join(_WORD.findall(t))


In [ ]:
%%writefile /kaggle/working/src/bartaguard/redflags.py
"""Explainable red-flag rules: transparent indicators, not proof of fraud.

Each rule returns the exact character spans that triggered it, so the apps can
highlight them. Patterns cover Bangla script, Banglish (romanised Bangla) and
English. The same rule list is exported to JSON for the phone apps, so the
patterns are kept to a regex dialect shared by Python, Swift and Kotlin
(no lookbehind, no possessive quantifiers, no inline flags).
"""
from __future__ import annotations

import json
import re
import unicodedata
from dataclasses import dataclass
from urllib.parse import urlsplit

SHORTENERS = {"bit.ly", "tinyurl.com", "t.co", "goo.gl", "is.gd", "cutt.ly", "rb.gy", "s.id", "shorturl.at",
              "ow.ly", "tiny.cc", "rebrand.ly", "t.ly", "v.gd", "bitly.com"}
RISKY_TLDS = {"xyz", "top", "click", "buzz", "icu", "cfd", "sbs", "rest", "monster", "live", "win", "bet", "vip"}
# Official domains of commonly impersonated brands; anything else using the brand name is flagged.
BRANDS = {"bkash": ["bkash.com"], "nagad": ["nagad.com.bd"], "rocket": ["dutchbanglabank.com"],
          "grameenphone": ["grameenphone.com"], "robi": ["robi.com.bd"], "banglalink": ["banglalink.net"]}


@dataclass(frozen=True)
class Rule:
    key: str
    severity: int          # 1 = context, 2 = warning, 3 = strong warning
    en: str
    bn: str
    patterns: tuple


RULES = [
    Rule("credential_request", 3,
         "Asks for an OTP, PIN, password or verification code. No bank, MFS or company needs these.",
         "OTP, পিন, পাসওয়ার্ড বা ভেরিফিকেশন কোড চাইছে। কোনো ব্যাংক, MFS বা প্রতিষ্ঠান এগুলো চায় না।",
         (r"\b(otp|o\.t\.p|pin|password|passcode|verification code|cvv)\b",
          r"(ওটিপি|পিন নম্বর|পিন কোড|গোপন পিন|পাসওয়ার্ড|ভেরিফিকেশন কোড|কোডটি দিন|কোড দিন|কোডটি বলুন|সিভিভি)",
          r"(পিন|pin)\s*(\S+\s*){0,3}(দিয়ে|দিন|বলুন|বলেন|পাঠান|din|bolun|bolen)",
          r"\b(otp|pin|code)\s*(ta|ti|ti\s*din|din|bolen|bolun|pathan|share)\b")),
    Rule("account_threat", 3,
         "Claims your account will be blocked, suspended or closed. A classic pressure tactic.",
         "আপনার অ্যাকাউন্ট বন্ধ/ব্লক হবে বলে ভয় দেখাচ্ছে, এটি একটি পরিচিত চাপ সৃষ্টির কৌশল।",
         (r"(account|acc)\s*(will be\s*)?(blocked|suspended|closed|locked|deactivated)",
          r"(অ্যাকাউন্ট|একাউন্ট|অ্যাকাউন্টটি|একাউন্টটি)\s*(\S+\s*){0,3}(বন্ধ|ব্লক|স্থগিত|লক)",
          r"(account|acc|একাউন্ট)\s*(bondho|block|lock)\s*(hoye|hobe|kora)")),
    Rule("advance_payment", 3,
         "Asks you to pay or send money first (fee, deposit, registration, processing) to receive a job, prize or refund.",
         "চাকরি, পুরস্কার বা রিফান্ড পেতে আগে টাকা/ফি/জামানত পাঠাতে বলছে।",
         (r"(registration|processing|delivery|security)\s*(fee|charge|deposit)",
          r"(রেজিস্ট্রেশন|রেজিষ্ট্রেশন|প্রসেসিং|ডেলিভারি|জামানত|অগ্রিম)\s*(ফি|চার্জ|টাকা)?",
          r"(send|pathan|pathiye|পাঠান|পাঠিয়ে|সেন্ড)\s*(\S+\s*){0,3}(taka|tk|টাকা)",
          r"(টাকা|taka)\s*(\S+\s*){0,2}(পাঠান|পাঠিয়ে দিন|send kor|pathan)")),
    Rule("wrong_transfer_refund", 3,
         "Says money was sent to you by mistake and asks you to send it back. Check your own balance in the official app first; this is a common scam.",
         "ভুল করে টাকা পাঠানো হয়েছে বলে ফেরত চাইছে। আগে অফিসিয়াল অ্যাপে নিজের ব্যালেন্স দেখুন; এটি একটি প্রচলিত প্রতারণা।",
         (r"ভুল\s*(করে|বশত)\s*(\S+\s*){0,6}(পাঠিয়েছি|চলে গেছে|গেছে|এসেছে)",
          r"(ফেরত|ferot|ferat|return)\s*(\S+\s*){0,2}(দিন|পাঠান|din|pathan|diben|koren)",
          r"\b(vul|bhul)\s*(kore|e)\s*(\S+\s*){0,5}(chole geche|pathiyechi|pathaisi|geche)",
          r"\b(sent|transferred)\s*(\S+\s*){0,3}by mistake")),
    Rule("prize_lottery", 2,
         "Says you have won a prize, lottery or reward you did not enter for.",
         "আপনি পুরস্কার/লটারি জিতেছেন বলছে, যেটিতে আপনি অংশ নেননি।",
         (r"\b(you (have )?won|winner|congratulations|lottery|jackpot|prize|reward)\b",
          r"(অভিনন্দন|বিজয়ী|জিতেছেন|লটারি|পুরস্কার|পুরষ্কার|উপহার জিতে)",
          r"\b(jitechen|bijoyi|lottery jitechen|puroskar)\b")),
    Rule("easy_income", 2,
         "Promises easy or guaranteed income, daily earnings or work-from-home money.",
         "সহজ বা নিশ্চিত আয়, দৈনিক ইনকাম বা ঘরে বসে টাকা আয়ের প্রতিশ্রুতি দিচ্ছে।",
         (r"\b(earn|income|profit)\b\s*(\S+\s*){0,3}(daily|per day|guaranteed|from home)",
          r"(ঘরে বসে|দৈনিক|প্রতিদিন|নিশ্চিত)\s*(\S+\s*){0,3}(আয়|ইনকাম|ইনকাম করুন|টাকা আয়)",
          r"(ইনকাম|income)\s*(\S+\s*){0,3}(করুন|korun|kora jay|করা যায়)",
          r"\b(ghore bose|ghore boshe)\b")),
    Rule("gambling", 2,
         "Mentions betting, casino or gambling. Common in scam and fraud campaigns.",
         "বেটিং, ক্যাসিনো বা জুয়ার কথা বলছে, যা প্রতারণামূলক প্রচারণায় প্রচলিত।",
         (r"\b(bet|betting|casino|1xbet|jackpot|spin|slot|aviator|crazy time)\b",
          r"(বেটিং|বাজি|ক্যাসিনো|জুয়া|স্লট|স্পিন)")),
    Rule("urgency", 1,
         "Creates urgency (\"now\", \"within 24 hours\", \"last chance\"). Slow down and verify.",
         "তাড়াহুড়ো তৈরি করছে (\"এখনই\", \"২৪ ঘণ্টার মধ্যে\", \"শেষ সুযোগ\")। ধীরে যাচাই করুন।",
         (r"\b(urgent|immediately|right now|last chance|within \d+ (hours?|minutes?)|expires? today)\b",
          r"(জরুরি|অবিলম্বে|শেষ সুযোগ|\d+\s*ঘণ্টার মধ্যে|এখনই\s*(\S+\s*){0,2}(না হলে|নইলে))",
          r"\b(ekhoni|joruri|druto)\b")),
    Rule("impersonation", 1,
         "Uses the name of a bank, mobile financial service, telecom or government office. Check it through official channels.",
         "ব্যাংক, মোবাইল ব্যাংকিং, টেলিকম বা সরকারি দপ্তরের নাম ব্যবহার করছে। অফিসিয়াল মাধ্যমে যাচাই করুন।",
         (r"\b(bkash|b-kash|nagad|rocket|upay|bank|police|govt|government|bangladesh bank)\b",
          r"(বিকাশ|নগদ|রকেট|উপায়|ব্যাংক|পুলিশ|সরকার|বাংলাদেশ ব্যাংক)")),
    Rule("callback_number", 1,
         "Asks you to call or message a number given in the message. Use the official number instead.",
         "মেসেজে দেওয়া নম্বরে কল/মেসেজ করতে বলছে। বরং অফিসিয়াল নম্বর ব্যবহার করুন।",
         (r"(call|whatsapp|imo|telegram|contact)\s*(\S+\s*){0,2}(0|\+?880)1[3-9]\d",
          r"(কল করুন|যোগাযোগ করুন|হোয়াটসঅ্যাপ|ইমো)\s*(\S+\s*){0,2}(0|\+?880|০|৮৮০)(1|১)")),
]

_URL = re.compile(r"(https?://[^\s<>\"']+|www\.[^\s<>\"']+|\b[a-z0-9-]+(\.[a-z0-9-]+)*\.(com|net|org|xyz|top|click|link|info|bd|me|io|ly|gl|co|cc|gd|at|live|win|bet|vip|buzz|icu|site|online|shop)(/[^\s<>\"']*)?)",
                  re.IGNORECASE)


def _norm(text: str) -> str:
    return unicodedata.normalize("NFC", text)


def find_urls(text: str) -> list:
    return [{"url": m.group(0), "start": m.start(), "end": m.end()} for m in _URL.finditer(_norm(text))]


def inspect_url(url: str) -> dict:
    """Local heuristics only: parse, never fetch."""
    u = url if "://" in url else "http://" + url
    parts = urlsplit(u)
    host = (parts.hostname or "").lower()
    flags = []
    if host in SHORTENERS:
        flags.append("shortener")
    if host.startswith("xn--") or ".xn--" in host:
        flags.append("punycode")
    if re.fullmatch(r"\d+\.\d+\.\d+\.\d+", host):
        flags.append("ip_address")
    tld = host.rsplit(".", 1)[-1] if "." in host else ""
    if tld in RISKY_TLDS:
        flags.append("risky_tld")
    if parts.scheme == "http":
        flags.append("not_https")
    for brand, official in BRANDS.items():
        if brand in host.replace("-", "") and not any(host == d or host.endswith("." + d) for d in official):
            flags.append(f"brand_lookalike:{brand}")
    if "@" in parts.netloc:
        flags.append("userinfo_trick")
    return {"url": url, "host": host, "flags": flags,
            "note": "Local heuristics only; the link was not opened or checked against a threat database."}


# Genuine OTP deliveries warn you not to share the code; that is a protective sign.
DONT_SHARE = (r"(do not|don't|never)\s+(share|tell|give)",
              r"(কাউকে|কারো সাথে|কারও সাথে)\s*(\S+\s*){0,3}(শেয়ার|দিবেন না|বলবেন না|জানাবেন না)",
              r"(share|kauke)\s*(\S+\s*){0,2}(korben na|diben na|bolben na)")


def scan(text: str) -> list:
    t = _norm(text)
    found = []
    dont_share = any(re.search(p, t, flags=re.IGNORECASE) for p in DONT_SHARE)
    for r in RULES:
        spans = []
        for p in r.patterns:
            for m in re.finditer(p, t, flags=re.IGNORECASE):
                if m.end() > m.start():
                    spans.append({"start": m.start(), "end": m.end(), "text": m.group(0)})
        if spans:
            if r.key == "credential_request" and dont_share:
                found.append({"key": "code_delivery", "severity": 0,
                              "en": "Looks like a code being sent to you with a warning not to share it. Never give this code to anyone who calls or messages you.",
                              "bn": "মনে হচ্ছে আপনাকে একটি কোড পাঠানো হয়েছে এবং শেয়ার না করতে সতর্ক করা হয়েছে। কেউ ফোন বা মেসেজ করে চাইলে এই কোড দেবেন না।",
                              "spans": spans})
                continue
            found.append({"key": r.key, "severity": r.severity, "en": r.en, "bn": r.bn, "spans": spans})
    for u in find_urls(t):
        info = inspect_url(u["url"])
        if info["flags"]:
            found.append({"key": "suspicious_link", "severity": 2 if set(info["flags"]) - {"not_https"} else 1,
                          "en": f"Link looks suspicious ({', '.join(info['flags'])}). Do not open it; type the official address yourself.",
                          "bn": f"লিংকটি সন্দেহজনক ({', '.join(info['flags'])})। এটি খুলবেন না; অফিসিয়াল ঠিকানা নিজে টাইপ করুন।",
                          "spans": [{"start": u["start"], "end": u["end"], "text": u["url"]}], "url": info})
    return sorted(found, key=lambda f: -f["severity"])


def export_rules() -> str:
    """JSON for the phone apps (same regexes, case-insensitive)."""
    return json.dumps({"rules": [{"key": r.key, "severity": r.severity, "en": r.en, "bn": r.bn,
                                  "patterns": list(r.patterns)} for r in RULES],
                       "dont_share_patterns": list(DONT_SHARE),
                       "shorteners": sorted(SHORTENERS), "risky_tlds": sorted(RISKY_TLDS),
                       "brands": BRANDS, "url_pattern": _URL.pattern}, ensure_ascii=False, indent=1)


In [ ]:
%%writefile /kaggle/working/src/bartaguard/verdict.py
"""Combine the model and the red-flag rules into the verdict users see.

Order of decisions (same logic is ported to Swift and Kotlin):
1. Text that is mostly Latin script is outside what the BTTC-trained model has
   seen (it learned "Latin letters = Telegram spam"), so the model abstains.
2. Any severity-3 rule (OTP/PIN request, account threat, advance fee, refund
   scam) -> "suspicious", whatever the model says.
3. The model's label is used only if its top probability clears the threshold
   picked on validation and no severity-2+ rule fired against a non-spam label.
4. Everything else -> "uncertain" ("Unable to assess reliably").
Severity-2 rules with an abstaining/uncertain model also give "suspicious".
"""
from __future__ import annotations

import re

LATIN = re.compile(r"[A-Za-z]")
BANGLA = re.compile(r"[ঀ-৿]")
MIN_BANGLA_SHARE = 0.5
MIN_LETTERS = 12


def bangla_share(text: str) -> float:
    b, l = len(BANGLA.findall(text)), len(LATIN.findall(text))
    return b / (b + l) if b + l else 0.0


def in_domain(text: str) -> bool:
    letters = len(BANGLA.findall(text)) + len(LATIN.findall(text))
    return letters >= MIN_LETTERS and bangla_share(text) >= MIN_BANGLA_SHARE


def decide(text: str, proba: dict | None, flags: list, threshold: float) -> dict:
    """proba: {"HAM": p, "PROMO": p, "SPAM": p} or None. Returns verdict + reasons."""
    top_sev = max((f["severity"] for f in flags), default=0)
    reasons = []
    model_label, model_conf = None, None
    if proba is not None and in_domain(text):
        model_label = max(proba, key=proba.get)
        model_conf = proba[model_label]
    elif proba is not None:
        reasons.append("out_of_domain")

    if top_sev >= 3:
        verdict = "suspicious"
        reasons.append("strong_red_flag")
    elif model_label is not None and model_conf >= threshold:
        if model_label == "SPAM":
            verdict = "suspicious"
            reasons.append("model_spam")
        elif top_sev >= 2:
            verdict = "suspicious"
            reasons.append("red_flag_overrides_model")
        else:
            verdict = "normal" if model_label == "HAM" else "promotional"
            reasons.append("model_confident")
    elif top_sev >= 2:
        verdict = "suspicious"
        reasons.append("red_flag")
    else:
        verdict = "uncertain"
        reasons.append("low_confidence" if model_label is not None else "no_model_opinion")
    return {"verdict": verdict, "model_label": model_label, "model_confidence": model_conf,
            "max_severity": top_sev, "reasons": reasons}


VERDICT_TO_LABEL = {"normal": "HAM", "promotional": "PROMO", "suspicious": "SPAM"}


In [ ]:
%%writefile /kaggle/working/src/bartaguard/transformer.py
"""Fine-tune Bangla-capable transformers on the BTTC splits (GPU: Kaggle T4 or Apple MPS).

    python -m bartaguard.transformer --data DIR --out ml/artifacts --models sagorsarker/bangla-bert-base

Per checkpoint: train with class-balanced loss, keep the epoch with the best
validation macro-F1, pick the abstention threshold on validation, then score
the test split (overall, SMS, Telegram) and the challenge set once.
"""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

LABELS = ["HAM", "PROMO", "SPAM"]
L2I = {l: i for i, l in enumerate(LABELS)}


def device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def encode(tok, texts, max_len):
    e = tok(list(texts), truncation=True, max_length=max_len, padding="max_length", return_tensors="pt")
    return e["input_ids"], e["attention_mask"]


@torch.no_grad()
def predict(model, ids, mask, dev, bs=128):
    model.eval()
    out = []
    for i in range(0, len(ids), bs):
        logits = model(input_ids=ids[i:i + bs].to(dev), attention_mask=mask[i:i + bs].to(dev)).logits
        out.append(torch.softmax(logits.float(), -1).cpu().numpy())
    return np.concatenate(out)


def metrics(y, proba):
    from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_recall_fscore_support
    pred = np.array(LABELS)[proba.argmax(1)]
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=LABELS, zero_division=0)
    return {"n": int(len(y)), "macro_f1": float(f1_score(y, pred, average="macro")), "accuracy": float((pred == y).mean()),
            "per_class_f1": {l: float(f[i]) for i, l in enumerate(LABELS)},
            "spam_recall": float(r[2]), "spam_precision": float(p[2]),
            "spam_pr_auc": float(average_precision_score(y == "SPAM", proba[:, 2])) if (y == "SPAM").any() else None,
            "legit_flagged_as_spam_rate": float(((pred == "SPAM") & (y != "SPAM")).sum() / max((y != "SPAM").sum(), 1)),
            "confusion": confusion_matrix(y, pred, labels=LABELS).tolist()}


def pick_threshold(y, proba, target=0.95):
    conf, ok = proba.max(1), np.array(LABELS)[proba.argmax(1)] == y
    for t in np.round(np.arange(0.34, 0.99, 0.01), 2):
        k = conf >= t
        if k.sum() and ok[k].mean() >= target:
            return float(t)
    return 0.99


def run(ckpt, tr, va, te, ch, out_dir, epochs=4, lr=3e-5, bs=32, max_len=160, log=print):
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
    torch.manual_seed(13)
    dev = device()
    tok = AutoTokenizer.from_pretrained(ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=3, id2label=dict(enumerate(LABELS)),
                                                               label2id=L2I).to(dev)
    enc = {k: encode(tok, d.text, max_len) for k, d in (("tr", tr), ("va", va), ("te", te), ("ch", ch))}
    ytr = torch.tensor(tr.label.map(L2I).values)
    dl = DataLoader(TensorDataset(*enc["tr"], ytr), batch_size=bs, shuffle=True)
    counts = np.bincount(ytr.numpy(), minlength=3)
    weights = torch.tensor(len(ytr) / (3 * counts), dtype=torch.float32, device=dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * epochs * len(dl)), epochs * len(dl))
    lossf = torch.nn.CrossEntropyLoss(weight=weights)
    use_amp = dev.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best, hist = (-1, None), []
    safe = ckpt.replace("/", "__")
    for ep in range(epochs):
        model.train()
        t0, tot = time.time(), 0.0
        for ids, mask, y in dl:
            opt.zero_grad()
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                loss = lossf(model(input_ids=ids.to(dev), attention_mask=mask.to(dev)).logits.float(), y.to(dev))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item() * len(y)
        pv = predict(model, *enc["va"], dev)
        f = metrics(va.label.values, pv)["macro_f1"]
        hist.append({"epoch": ep + 1, "train_loss": tot / len(ytr), "val_macro_f1": f, "sec": round(time.time() - t0)})
        log(f"{ckpt} epoch {ep + 1} loss={tot / len(ytr):.4f} val macroF1={f:.4f} ({time.time() - t0:.0f}s)")
        if f > best[0]:
            best = (f, ep + 1)
            model.save_pretrained(out_dir / safe)
            tok.save_pretrained(out_dir / safe)
    model = AutoModelForSequenceClassification.from_pretrained(out_dir / safe).to(dev)
    pv, pt, pc = (predict(model, *enc[k], dev) for k in ("va", "te", "ch"))
    thr = pick_threshold(va.label.values, pv)
    keep = pt.max(1) >= thr
    res = {"model": ckpt, "best_epoch": best[1], "val_macro_f1": best[0], "history": hist, "threshold": thr,
           "test": metrics(te.label.values, pt),
           "test_sms": metrics(te[te.source == "SMS"].label.values, pt[(te.source == "SMS").values]),
           "test_telegram": metrics(te[te.source == "Telegram"].label.values, pt[(te.source == "Telegram").values]),
           "test_abstention": {"coverage": float(keep.mean()),
                               "accuracy_when_answered": float((np.array(LABELS)[pt[keep].argmax(1)] == te.label.values[keep]).mean()) if keep.any() else None},
           "challenge": metrics(ch.label.values, pc),
           "challenge_by_script": {s: float((np.array(LABELS)[pc[(ch.script == s).values].argmax(1)] == ch.label.values[(ch.script == s).values]).mean())
                                   for s in ch.script.unique()}}
    pd.DataFrame(pt, columns=LABELS).assign(text=te.text.values, label=te.label.values, source=te.source.values) \
        .to_parquet(out_dir / f"test_proba_{safe}.parquet", index=False)
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", type=Path, required=True)
    ap.add_argument("--out", type=Path, default=Path("ml/artifacts"))
    ap.add_argument("--models", nargs="+", default=["sagorsarker/bangla-bert-base", "FacebookAI/xlm-roberta-base"])
    ap.add_argument("--epochs", type=int, default=4)
    a = ap.parse_args()
    a.out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(a.data / "bttc_splits.csv")
    ch = pd.read_csv(a.data / "challenge_set.csv")
    tr, va, te = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
    results = []
    for ckpt in a.models:
        r = run(ckpt, tr, va, te, ch, a.out, epochs=a.epochs)
        results.append(r)
        print(json.dumps({k: r[k] for k in ("model", "best_epoch", "val_macro_f1")}),
              "test macroF1", round(r["test"]["macro_f1"], 4), "SMS", round(r["test_sms"]["macro_f1"], 4),
              "challenge acc", round(r["challenge"]["accuracy"], 3), r["challenge_by_script"], flush=True)
        (a.out / "metrics_transformers.json").write_text(json.dumps({"results": results, "device": str(device()),
                                                                     "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
import glob, os, torch
hits = glob.glob('/kaggle/input/**/bttc_splits.csv', recursive=True)
assert hits, 'Attach the BartaGuard dataset (bttc_splits.csv)'
DATA = os.path.dirname(hits[0]); print(DATA, os.listdir(DATA))
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
%cd /kaggle/working/src
!python -m bartaguard.transformer --data {DATA} --out /kaggle/working/artifacts --models sagorsarker/bangla-bert-base FacebookAI/xlm-roberta-base

In [ ]:
import json, pandas as pd
M = json.load(open('/kaggle/working/artifacts/metrics_transformers.json'))
pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, (dict, list))} for r in M['results']])

In [ ]:
!cd /kaggle/working && rm -rf artifacts/*/checkpoint-* && zip -qr bartaguard_artifacts.zip artifacts && ls -la /kaggle/working